# Stage Silver — MSPAS Morbilidad

Transforma las 3 tablas Bronze de MSPAS hacia `stage_silver_ss2`.

| Tabla Bronze | Tabla Silver | Descripción |
|---|---|---|
| `dbx_primeras_causas_de_morbilidad_2015_a_2025` | misma | 20 primeras causas de morbilidad por municipio |
| `dbx_morbilidad_enfermedades_cronicas_2015_a_2025` | misma | Enfermedades crónicas por municipio |
| `dbx_morbilidad_grupo_materno_infantil_2012_a_2025` | misma | Morbilidad materno-infantil |

**Transformaciones aplicadas:**
- `anio` → INT (enfermedades_cronicas puede llegar como DOUBLE)
- `casos_total` = COALESCE(casos, cantidad) — resuelve nulls cruzados; luego se eliminan `casos` y `cantidad`
- `cie_10_norm` = regexp_replace(cie_10, '[: ]', '') — elimina separadores; se conserva `cie_10` original
- `departamento` / `municipio` → TRIM
- `departamento_oficial` → JOIN con catálogo ref (nombre normalizado)
- `dropDuplicates()` — reintentos de ingesta generan duplicados
- Auditoría Silver: `silver_processed_timestamp`, `silver_job_run_id`

In [0]:
# ── Setup: imports, config, catálogo y helpers en una sola celda ─────────────
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

BRONZE_SCHEMA = "sandbox_bronze_ss2"
SILVER_SCHEMA = "stage_silver_ss2"
WRITE_MODE    = "overwrite"

# ── Catálogo de referencia: 22 departamentos oficiales de Guatemala ───────────
# JOIN sobre UPPER(TRIM(col_depto)) == depto_variante para tolerar cualquier capitalización.
_DEPTO_ROWS = [
    ("ALTA VERAPAZ",            "Alta Verapaz",          ),
    ("BAJA VERAPAZ",            "Baja Verapaz",          ),
    ("CHIMALTENANGO",           "Chimaltenango",         ),
    ("CHIQUIMULA",              "Chiquimula",            ),
    ("EL PROGRESO",             "El Progreso",           ),
    ("ESCUINTLA",               "Escuintla",             ),
    ("GUATEMALA",               "Guatemala",             ),
    ("HUEHUETENANGO",           "Huehuetenango",         ),
    ("IZABAL",                  "Izabal",                ),
    ("JALAPA",                  "Jalapa",                ),
    ("JUTIAPA",                 "Jutiapa",               ),
    ("PETEN",                   "Petén",                 ),
    ("EL PETEN",                "Petén",                 ),
    ("EL PETÉN",                "Petén",                 ),
    ("PETÉN",                   "Petén",                 ),
    ("QUETZALTENANGO",          "Quetzaltenango",        ),
    ("QUICHE",                  "El Quiché",             ),
    ("QUICHÉ",                  "El Quiché",             ),
    ("EL QUICHE",               "El Quiché",             ),
    ("EL QUICHÉ",               "El Quiché",             ),
    ("RETALHULEU",              "Retalhuleu",            ),
    ("SACATEPEQUEZ",            "Sacatepéquez",          ),
    ("SACATEPÉQUEZ",            "Sacatepéquez",          ),
    ("SAN MARCOS",              "San Marcos",            ),
    ("SANTA ROSA",              "Santa Rosa",            ),
    ("SOLOLA",                  "Sololá",                ),
    ("SOLOLÁ",                  "Sololá",                ),
    ("SUCHITEPEQUEZ",           "Suchitepéquez",         ),
    ("SUCHITEPÉQUEZ",           "Suchitepéquez",         ),
    ("TOTONICAPAN",             "Totonicapán",           ),
    ("TOTONICAPÁN",             "Totonicapán",           ),
    ("ZACAPA",                  "Zacapa",                ),
    ("TODOS LOS DEPARTAMENTOS", "Todos los departamentos"),
]

ref_depto_df = spark.createDataFrame(
    _DEPTO_ROWS,
    ["depto_variante", "depto_oficial"]
)

# ── Helpers ───────────────────────────────────────────────────────────────────

def get_job_run_id():
    try:
        return (
            dbutils.notebook.entry_point
            .getDbutils().notebook().getContext()
            .currentRunId().toString()
        )
    except Exception:
        return "manual"


def normalize_cie10(col_expr):
    """'Otras causas' → 'R99'  |  J:18:0 → J180  |  B:37 → B37"""
    return (
        F.when(col_expr.isin("Todas las causas", "Todas las causas externas"), F.lit("A00-Y98"))
         .when(col_expr == "Otras causas", F.lit("R99"))
         .otherwise(F.regexp_replace(col_expr, r'[: ]', ''))
    )


def join_departamento(df, depto_col, ref_df):
    """
    LEFT JOIN sobre UPPER(TRIM(depto_col)).
    Agrega departamento_oficial (nombre normalizado).
    Filas sin match quedan con NULL — no se descartan.
    El ID numérico lo asigna Gold al construir dim_geografia, no Silver.
    """
    return (
        df
        .join(
            F.broadcast(ref_df),
            F.upper(F.trim(F.col(depto_col))) == F.col("depto_variante"),
            "left",
        )
        .drop("depto_variante")
        .withColumnRenamed("depto_oficial", "departamento_oficial")
    )


def add_silver_audit(df, run_id):
    return (
        df
        .withColumn("silver_processed_timestamp", F.current_timestamp())
        .withColumn("silver_job_run_id",           F.lit(run_id))
    )


def ensure_column(df, col_name, dtype):
    """Agrega la columna como NULL si no existe en el DataFrame."""
    if col_name not in df.columns:
        df = df.withColumn(col_name, F.lit(None).cast(dtype))
    return df


RUN_ID = get_job_run_id()
logger.info(f"Setup OK — run_id={RUN_ID}")

## Tabla 1: dbx_primeras_causas_de_morbilidad_2015_a_2025
Columnas Bronze: `anio, departamento, municipio, cie_10, diagnostico, grupo_etario, sexo, cantidad, casos`  
`cantidad` llega NULL en esta tabla; `casos` contiene el valor real.

In [0]:
TABLE      = "dbx_primeras_causas_de_morbilidad_2015_a_2025"
bronze_tbl = f"{BRONZE_SCHEMA}.{TABLE}"
silver_tbl = f"{SILVER_SCHEMA}.{TABLE}"

df = spark.table(bronze_tbl)
logger.info(f"[{TABLE}] Bronze: {df.count():,} filas")

df = ensure_column(df, "casos",    "double")
df = ensure_column(df, "cantidad", "double")

df_silver = (
    df
    .withColumn("anio",         F.col("anio").cast(IntegerType()))
    .withColumn("casos_total",  F.coalesce(F.col("casos"), F.col("cantidad")).cast(LongType()))
    .drop("casos", "cantidad")
    .withColumn("cie_10_norm",  normalize_cie10(F.col("cie_10")))
    .withColumn("departamento", F.trim(F.col("departamento")))
    .withColumn("municipio",    F.trim(F.col("municipio")))
    .dropDuplicates()
)
df_silver = join_departamento(df_silver, "departamento", ref_depto_df)
df_silver = add_silver_audit(df_silver, RUN_ID)

logger.info(f"[{TABLE}] Silver: {df_silver.count():,} filas")

(
    df_silver.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(silver_tbl)
)
logger.info(f"Escrito → {silver_tbl}")

## Tabla 2: dbx_morbilidad_enfermedades_cronicas_2015_a_2025
Columnas Bronze: `anio, departamento, municipio, cie_10, diagnostico, grupo_etario, sexo, casos`  
`anio` puede ser DOUBLE en esta tabla (detectado en Bronze). No tiene columna `cantidad`.

In [0]:
TABLE      = "dbx_morbilidad_enfermedades_cronicas_2015_a_2025"
bronze_tbl = f"{BRONZE_SCHEMA}.{TABLE}"
silver_tbl = f"{SILVER_SCHEMA}.{TABLE}"

df = spark.table(bronze_tbl)
logger.info(f"[{TABLE}] Bronze: {df.count():,} filas")

df = ensure_column(df, "casos",    "double")
df = ensure_column(df, "cantidad", "double")

df_silver = (
    df
    .withColumn("anio",         F.col("anio").cast(IntegerType()))  # DOUBLE → INT
    .withColumn("casos_total",  F.coalesce(F.col("casos"), F.col("cantidad")).cast(LongType()))
    .drop("casos", "cantidad")
    .withColumn("cie_10_norm",  normalize_cie10(F.col("cie_10")))
    .withColumn("departamento", F.trim(F.col("departamento")))
    .withColumn("municipio",    F.trim(F.col("municipio")))
    .dropDuplicates()
)
df_silver = join_departamento(df_silver, "departamento", ref_depto_df)
df_silver = add_silver_audit(df_silver, RUN_ID)

logger.info(f"[{TABLE}] Silver: {df_silver.count():,} filas")

(
    df_silver.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(silver_tbl)
)
logger.info(f"Escrito → {silver_tbl}")

## Tabla 3: dbx_morbilidad_grupo_materno_infantil_2012_a_2025
Columnas Bronze: `anio, departamento, municipio, cie_10, diagnostico, grupo_etario, sexo, casos, cantidad`  
`cantidad` puede llegar NULL en algunos archivos mientras `casos` tiene el valor (y viceversa).

In [0]:
TABLE      = "dbx_morbilidad_grupo_materno_infantil_2012_a_2025"
bronze_tbl = f"{BRONZE_SCHEMA}.{TABLE}"
silver_tbl = f"{SILVER_SCHEMA}.{TABLE}"

df = spark.table(bronze_tbl)
logger.info(f"[{TABLE}] Bronze: {df.count():,} filas")

df = ensure_column(df, "casos",    "double")
df = ensure_column(df, "cantidad", "double")

df_silver = (
    df
    .withColumn("anio",         F.col("anio").cast(IntegerType()))
    .withColumn("casos_total",  F.coalesce(F.col("casos"), F.col("cantidad")).cast(LongType()))
    .drop("casos", "cantidad")
    .withColumn("cie_10_norm",  normalize_cie10(F.col("cie_10")))
    .withColumn("departamento", F.trim(F.col("departamento")))
    .withColumn("municipio",    F.trim(F.col("municipio")))
    .dropDuplicates()
)
df_silver = join_departamento(df_silver, "departamento", ref_depto_df)
df_silver = add_silver_audit(df_silver, RUN_ID)

logger.info(f"[{TABLE}] Silver: {df_silver.count():,} filas")

(
    df_silver.write
    .format("delta")
    .mode(WRITE_MODE)
    .option("overwriteSchema", "true")
    .saveAsTable(silver_tbl)
)
logger.info(f"Escrito → {silver_tbl}")

## Validación — Row counts Bronze vs Silver

In [0]:
_TABLES = [
    "dbx_primeras_causas_de_morbilidad_2015_a_2025",
    "dbx_morbilidad_enfermedades_cronicas_2015_a_2025",
    "dbx_morbilidad_grupo_materno_infantil_2012_a_2025",
]

print(f"{'Tabla':<55} {'Bronze':>10} {'Silver':>10} {'Estado':>8}")
print("-" * 87)
for t in _TABLES:
    b = spark.table(f"{BRONZE_SCHEMA}.{t}").count()
    s = spark.table(f"{SILVER_SCHEMA}.{t}").count()
    estado = "OK" if b == s else f"DIFF ({b - s:+,})"
    print(f"{t:<55} {b:>10,} {s:>10,} {estado:>8}")

print("\n── Muestra normalización CIE-10 ──")
(
    spark.table(f"{SILVER_SCHEMA}.dbx_primeras_causas_de_morbilidad_2015_a_2025")
    .select("cie_10", "cie_10_norm", "departamento", "departamento_oficial", "anio", "casos_total")
    .dropDuplicates(["cie_10"])
    .limit(10)
    .show(truncate=False)
)

print("── Departamentos sin match en catálogo ──")
for t in _TABLES:
    sin_match = (
        spark.table(f"{SILVER_SCHEMA}.{t}")
        .filter(F.col("departamento_oficial").isNull())
        .select("departamento").distinct()
    )
    cnt = sin_match.count()
    if cnt > 0:
        print(f"  {t}: {cnt} departamentos sin match — revisar catálogo")
        sin_match.show(truncate=False)
    else:
        print(f"  {t}: todos los departamentos resueltos OK")